# 🤖 Model Training — Energy Grid Outage Prediction

Trains a lightweight PySpark RandomForest classifier.
Optimized for Trial capacity (samples data, small model).

In [ ]:
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print('Imports OK')

In [ ]:
# Load features. Sample down to keep memory low on Trial capacity.
df = spark.read.table('gold_ml_features')
total = df.count()
print(f'Total feature rows: {total}')

# Cap training data at 25k rows to avoid session memory pressure
MAX_ROWS = 25000
if total > MAX_ROWS:
    df = df.sample(fraction=MAX_ROWS / total, seed=42)
    print(f'Sampled to ~{df.count()} rows')

In [ ]:
# Clean nulls/NaN in numeric columns
for c in df.columns:
    dtype = dict(df.dtypes).get(c, '')
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

df.groupBy('had_outage').count().show()

In [ ]:
# Build feature vector from numeric columns
feature_cols = [
    c for c in df.columns
    if dict(df.dtypes).get(c, '') in ('double', 'float', 'int', 'bigint', 'long')
    and c not in ('had_outage', 'event_count', 'total_affected', 'total_event_duration')
]
print(f'Features ({len(feature_cols)}): {feature_cols}')

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(df).select('features', col('had_outage').cast('double').alias('label'))
model_df = model_df.coalesce(2).cache()
print(f'Model rows: {model_df.count()}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count()}, Test: {test_df.count()}')

In [ ]:
# Small RandomForest — light on Trial capacity
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    rawPredictionCol='rawPrediction',
    probabilityCol='probability',
    numTrees=20,
    maxDepth=5,
    maxBins=32,
    seed=42,
)
model = rf.fit(train_df)
print('RandomForest model trained')

In [ ]:
predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy').evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1').evaluate(predictions)

print(f'AUC-ROC:  {auc:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1:       {f1:.4f}')

In [ ]:
metrics = spark.createDataFrame(
    [('energy-grid', 'outage-prediction', 'RandomForestClassifier',
      len(feature_cols), train_df.count(), test_df.count(),
      float(auc), float(accuracy), float(f1))],
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Metrics saved')
model_df.unpersist()

In [ ]:
model.write().overwrite().save('Files/models/outage_prediction_rf')
print('Model saved')

# 🤖 Model Training — Energy Grid Outage Prediction

Trains a PySpark RandomForest classifier to predict grid outages.

**Target:** `had_outage` (1 = outage/surge/sag event occurred)

In [ ]:
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print('Imports OK')

In [ ]:
# Load features and clean nulls/NaN
df = spark.read.table('gold_ml_features')
print(f'Raw rows: {df.count()}')
print(f'Columns: {df.columns}')

# Fill all nulls and NaNs with 0
for c in df.columns:
    dtype = dict(df.dtypes).get(c, '')
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

print(f'Cleaned rows: {df.count()}')
df.groupBy('had_outage').count().show()

In [ ]:
# Use only simple numeric features that are guaranteed to exist
feature_cols = [
    c for c in df.columns
    if dict(df.dtypes).get(c, '') in ('double', 'float', 'int', 'bigint', 'long')
    and c not in ('had_outage', 'event_count', 'total_affected', 'total_event_duration')
]
print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(df).select('features', col('had_outage').cast('double').alias('label'))
model_df = model_df.filter(col('label').isNotNull())
print(f'Model rows: {model_df.count()}')

In [ ]:
# Split
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count()}, Test: {test_df.count()}')

In [ ]:
# Train RandomForest (always available in PySpark, no SynapseML needed)
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    rawPredictionCol='rawPrediction',
    probabilityCol='probability',
    numTrees=50,
    maxDepth=8,
    seed=42,
)
model = rf.fit(train_df)
print('RandomForest model trained')

In [ ]:
# Evaluate
predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy').evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1').evaluate(predictions)

print(f'AUC-ROC:  {auc:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1:       {f1:.4f}')

In [ ]:
# Save metrics
metrics = spark.createDataFrame(
    [('energy-grid', 'outage-prediction', 'RandomForestClassifier',
      len(feature_cols), train_df.count(), test_df.count(),
      float(auc), float(accuracy), float(f1))],
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Metrics saved')

In [ ]:
# Save model
model.save('Files/models/outage_prediction_rf')
print('Model saved')

# 🤖 Model Training — Energy Grid Outage Prediction

Trains a classifier to predict grid outages based on sensor readings.
Uses SynapseML LightGBM with fallback to scikit-learn if unavailable.

**Target:** `had_outage` (1 = outage/surge/sag event occurred)

**Reads from:** `gold_ml_features`

**Writes to:** `gold_ml_model_metrics`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan, isnull
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load and clean features — fill nulls/NaNs that stddev() produces for single-value groups
features_df = spark.read.table('gold_ml_features')
print(f'Raw feature rows: {features_df.count():,}')

# Fill nulls in numeric columns with 0
numeric_cols = [c for c, t in features_df.dtypes if t in ('double', 'float', 'int', 'bigint', 'long')]
features_df = features_df.na.fill(0, numeric_cols)

# Also replace NaN with 0
for c in numeric_cols:
    features_df = features_df.withColumn(c, when(isnan(col(c)), 0).otherwise(col(c)))

print(f'Cleaned feature rows: {features_df.count():,}')
features_df.groupBy('had_outage').count().show()

In [ ]:
# Feature columns
numeric_features = [
    'avg_voltage', 'std_voltage', 'min_voltage', 'max_voltage', 'voltage_range', 'voltage_deviation',
    'avg_frequency', 'std_frequency', 'freq_deviation',
    'avg_power_factor', 'min_power_factor',
    'avg_load', 'max_load',
    'avg_temp', 'max_temp',
    'reading_count',
    'day_of_week', 'month',
]

# Only use features that exist in the dataframe
available = set(features_df.columns)
numeric_features = [f for f in numeric_features if f in available]
print(f'Available numeric features: {len(numeric_features)}')

# Index region
if 'region' in available:
    indexer_region = StringIndexer(inputCol='region', outputCol='region_idx', handleInvalid='keep')
    features_df = indexer_region.fit(features_df).transform(features_df)
    all_features = numeric_features + ['region_idx']
else:
    all_features = numeric_features

assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(features_df).select('features', col('had_outage').cast('double').alias('label'))
model_df = model_df.filter(col('label').isNotNull())
print(f'Model-ready rows: {model_df.count():,}')
print(f'Feature count: {len(all_features)}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} rows')
print(f'Test:  {test_df.count():,} rows')

In [ ]:
# Try SynapseML LightGBM first, fallback to scikit-learn
model = None
model_type = 'unknown'

try:
    from synapse.ml.lightgbm import LightGBMClassifier
    lgbm = LightGBMClassifier(
        featuresCol='features',
        labelCol='label',
        predictionCol='prediction',
        rawPredictionCol='rawPrediction',
        probabilityCol='probability',
        numLeaves=31,
        numIterations=100,
        learningRate=0.05,
        featureFraction=0.8,
        baggingFraction=0.8,
        baggingFreq=5,
        objective='binary',
        isUnbalance=True,
    )
    model = lgbm.fit(train_df)
    model_type = 'LightGBMClassifier'
    print('SynapseML LightGBM model trained')
except Exception as e:
    print(f'SynapseML LightGBM failed: {e}')
    print('Falling back to PySpark RandomForest...')
    from pyspark.ml.classification import RandomForestClassifier
    rf = RandomForestClassifier(
        featuresCol='features',
        labelCol='label',
        predictionCol='prediction',
        rawPredictionCol='rawPrediction',
        probabilityCol='probability',
        numTrees=100,
        maxDepth=10,
        seed=42,
    )
    model = rf.fit(train_df)
    model_type = 'RandomForestClassifier'
    print('PySpark RandomForest model trained (fallback)')

In [ ]:
predictions = model.transform(test_df)

auc_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
pr_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderPR')
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')

auc = auc_eval.evaluate(predictions)
auc_pr = pr_eval.evaluate(predictions)
accuracy = acc_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)

print(f'\n=== Model Evaluation ({model_type}) ===')
print(f'AUC-ROC:  {auc:.4f}')
print(f'AUC-PR:   {auc_pr:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')

In [ ]:
metrics_data = [
    ('energy-grid', 'outage-prediction', model_type,
     len(all_features), train_df.count(), test_df.count(),
     float(auc), float(auc_pr), float(accuracy), float(f1))
]
metrics_df = spark.createDataFrame(
    metrics_data,
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'auc_pr', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved')
metrics_df.show(truncate=False)

In [ ]:
# Save model
model.save('Files/models/outage_prediction_model')
print(f'Model saved ({model_type})')

# 🤖 Model Training — Energy Grid Outage Prediction

Trains a **SynapseML LightGBM** classifier to predict grid outages
based on sensor readings and historical event patterns.

**Target:** `had_outage` (1 = outage/surge/sag event occurred)

**Reads from:** `gold_ml_features`

**Writes to:** `gold_ml_model_metrics`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
features_df = spark.read.table('gold_ml_features')
print(f'Feature rows: {features_df.count():,}')
features_df.groupBy('had_outage').count().show()

In [ ]:
numeric_features = [
    'avg_voltage', 'std_voltage', 'min_voltage', 'max_voltage', 'voltage_range', 'voltage_deviation',
    'avg_frequency', 'std_frequency', 'freq_deviation',
    'avg_power_factor', 'min_power_factor',
    'avg_load', 'max_load',
    'avg_temp', 'max_temp',
    'reading_count',
    'day_of_week', 'month',
]

indexer_region = StringIndexer(inputCol='region', outputCol='region_idx', handleInvalid='keep')
indexed_df = indexer_region.fit(features_df).transform(features_df)

all_features = numeric_features + ['region_idx']

assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df).select('features', col('had_outage').alias('label'))
print(f'Model-ready rows: {model_df.count():,}')
print(f'Feature count: {len(all_features)}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} rows')
print(f'Test:  {test_df.count():,} rows')

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier

lgbm = LightGBMClassifier(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    rawPredictionCol='rawPrediction',
    probabilityCol='probability',
    numLeaves=31,
    numIterations=200,
    learningRate=0.05,
    featureFraction=0.8,
    baggingFraction=0.8,
    baggingFreq=5,
    objective='binary',
    isUnbalance=True,
)

model = lgbm.fit(train_df)
print('LightGBM model trained')

In [ ]:
predictions = model.transform(test_df)

auc_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
pr_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderPR')
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')

auc = auc_eval.evaluate(predictions)
auc_pr = pr_eval.evaluate(predictions)
accuracy = acc_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)

print(f'\n=== Model Evaluation ===')
print(f'AUC-ROC:  {auc:.4f}')
print(f'AUC-PR:   {auc_pr:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')

In [ ]:
metrics_data = [
    ('energy-grid', 'outage-prediction', 'LightGBMClassifier',
     len(all_features), train_df.count(), test_df.count(),
     float(auc), float(auc_pr), float(accuracy), float(f1))
]
metrics_df = spark.createDataFrame(
    metrics_data,
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'auc_pr', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved')
metrics_df.show(truncate=False)

In [ ]:
model.save('Files/models/outage_prediction_lgbm')
print('Model saved to Files/models/outage_prediction_lgbm')